# 🔤 OCR Worker — AIC 2026 (Chạy theo Lesson trên Colab GPU T4)
## Hướng dẫn:
1. Vào **Runtime → Change runtime type → T4 GPU**.
2. *(Nếu repo Private)*: Vào icon **chìa khóa 🔑 (Secrets)** bên trái Colab, thêm Secret tên `GITHUB_TOKEN` và bật Notebook access.
3. Điền các Lesson bạn phụ trách ở Cell Cấu hình bên dưới (ví dụ: `LESSONS = 'L21,L22,L23,L24'` hoặc `LESSONS = 'L25'`).
4. Chạy từng Cell từ trên xuống dưới.

In [ ]:
# ═══════════════════════════════════════════
# ⚙️ CẤU HÌNH LESSON BẠN PHỤ TRÁCH
# ═══════════════════════════════════════════
# Ví dụ phân công:
#   Người 1: LESSONS = 'L21,L22,L23,L24,L27,L28,L29,L30'
#   Người 2: LESSONS = 'L25'
#   Người 3: LESSONS = 'L26'
LESSONS = 'L21,L22,L23,L24'

# Thông tin Repo và Drive
GIT_REPO        = 'https://github.com/NGUYENHUUTINH-uit/AI_Challenge_2026.git'
GIT_BRANCH      = 'feature-A7'
DRIVE_DATA_PATH = '/content/drive/MyDrive/AIC2026'

## Bước 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive đã mount thành công!')

## Bước 2: Kéo code từ nhánh `feature-A7` về Colab

In [ ]:
import os
from google.colab import userdata

try:
    # Tự động lấy token từ mục Secrets (icon chìa khóa 🔑 bên trái Colab: Name = GITHUB_TOKEN)
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{GITHUB_TOKEN}@github.com/NGUYENHUUTINH-uit/AI_Challenge_2026.git"
except Exception:
    # Nếu repo là PUBLIC hoặc không cài Secrets:
    repo_url = GIT_REPO

if not os.path.exists('/content/AIC_2026'):
    !git clone -b {GIT_BRANCH} {repo_url} /content/AIC_2026
else:
    !cd /content/AIC_2026 && git pull origin {GIT_BRANCH}

os.chdir('/content/AIC_2026')
print('✅ Code sẵn sàng tại /content/AIC_2026')

## Bước 3: Unzip Keyframes từ Drive ra ổ đĩa Colab
> Chỉ cần unzip 1 lần. Nếu session trước đã unzip thì cell này sẽ tự động bỏ qua (Skip).

In [ ]:
import zipfile, os
from pathlib import Path

zip_dir = Path(DRIVE_DATA_PATH)
out_dir = Path('/content/keyframes')

# Kiểm tra nếu đã unzip từ trước
if out_dir.exists() and len(list(out_dir.iterdir())) > 0:
    print(f'✅ [Skip] /content/keyframes đã có sẵn {len(list(out_dir.iterdir()))} thư mục video.')
else:
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_files = sorted(zip_dir.glob('keyframes_L*.zip'))
    print(f'Tìm thấy {len(zip_files)} file zip: {[f.name for f in zip_files]}')
    
    for zf in zip_files:
        print(f'Đang giải nén {zf.name} ({zf.stat().st_size/1024/1024/1024:.2f} GB) ...')
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(out_dir)
        print(f'  ✅ Xong {zf.name}')

# Tạo symlink liên kết vào thư mục project
project_kf = Path('/content/AIC_2026/data/btc/Keyframes')
project_kf.parent.mkdir(parents=True, exist_ok=True)
if project_kf.is_symlink():
    project_kf.unlink()
if not project_kf.exists():
    os.symlink('/content/keyframes', str(project_kf))

print(f'\n✅ Hoàn tất! Đã liên kết {len(list(project_kf.iterdir()))} thư mục video vào project.')

## Bước 4: Cài EasyOCR & Xem danh sách thống kê Lesson

In [ ]:
!pip install easyocr -q
print('✅ Đã cài đặt EasyOCR!')

!cd /content/AIC_2026 && python -m src.indexing.run_ocr_worker --plan

## Bước 5: Chạy OCR cho các Lesson bạn chọn (`LESSONS`)
> ⏱️ Script **tự động lưu và resume** nếu bị ngắt kết nối — chỉ cần bấm chạy lại cell này.

In [ ]:
!cd /content/AIC_2026 && python -m src.indexing.run_ocr_worker --lessons {LESSONS}

## Bước 6: Copy file kết quả về Google Drive

In [ ]:
import shutil
from pathlib import Path

src_dir = Path('/content/AIC_2026/data/processed')
dst_dir = Path(DRIVE_DATA_PATH) / 'ocr_results'
dst_dir.mkdir(parents=True, exist_ok=True)

part_files = list(src_dir.glob('ocr_part_*.json'))
for f in part_files:
    dst_file = dst_dir / f.name
    shutil.copy2(f, dst_file)
    size_mb = dst_file.stat().st_size / 1024 / 1024
    print(f'✅ Đã lưu về Drive: {dst_file} ({size_mb:.1f} MB)')

if not part_files:
    print('❌ Chưa có file kết quả ocr_part_*.json nào.')
else:
    print('\n👉 Bạn có thể tải các file trên từ Drive về máy để gộp chung với nhóm!')